In [ ]:
import cv2
import numpy as np
import serial
import serial
import time

In [ ]:
port='/dev/rfcomm0'
baud_rate=230400
ser=serial.Serial(port, baud_rate, timeout=1)
print("Connected:", ser.is_open) 

In [ ]:
size=20
display_len=np.linspace(0,size//2-1,size//2).astype(np.uint16)
start=0
rot=20
num=2

In [ ]:
path="video.mp4"
video=cv2.VideoCapture(path)
transmit=bytearray([4,rot,0,0])
ser.write(transmit)
while video.isOpened():
    _,frame=video.read()
    if (_==0):
        break
    image=cv2.resize(frame,(size,size),interpolation=cv2.INTER_AREA)
    theta=0
    img_mask=np.zeros((size,size,3)).astype(np.uint8)
    if (ser.is_open==False):
        print("disconnected")
        break
    while(theta<360):
        theta=theta+1
        coordi_x=(size//2+display_len*np.cos(theta*np.pi/180)).astype(np.int16)
        coordi_y=(size//2+display_len*np.sin(theta*np.pi/180)).astype(np.int16)
        for i in range (0,coordi_y.size):
            start+=1
            if (start>16):
                start=0
                break
            x=coordi_x[i]
            y=coordi_y[i]
            b,g,r=image[y,x]
            if (start==2):
                num=1
            elif (start==9):
                num=2
            else:
                num=3
            transmit=bytearray([num,r,g,b])
            ser.write(transmit)
            img_mask[y,x]=image[y,x]
    cv2.imshow("image",img_mask)
    if cv2.waitKey(1)&0xff==ord(" "):
        break
cv2.destroyAllWindows()
video.release()    

In [ ]:
cv2.destroyAllWindows()
video.release() 